# Exit-policy study, phase F: spot-led versus perp-led extremes

**Question (BACKLOG research item 2, row F of the exhaustion brainstorm):** does a new high carried by perpetual takers
and a rising premium, while Binance spot takers lag, tell us anything about what the rest of the trade is worth?

This notebook is a check on the frozen run, not a second run. It reloads `results/spot_perp/events.csv.gz` and
recomputes each decision line's mean, interval and halves with the same bootstrap, then asserts they match
`report.json`. Design: [PREREGISTRATION_SPOT_PERP.md](PREREGISTRATION_SPOT_PERP.md) v1.1 plus amendment A26. Verdict:
[findings_spot_perp.md](findings_spot_perp.md).

In [1]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
HERE = Path.cwd(); sys.path.insert(0, str(HERE))
import micro_lib as M, spotperp_lib as L
R = HERE / "results" / "spot_perp"
report = json.loads((R / "report.json").read_text())
verdict = json.loads((R / "verdict.json").read_text())
f0 = json.loads((R / "freeze_F0.json").read_text())
pre = json.loads((R / "preconditions.json").read_text())
events = pd.read_csv(R / "events.csv.gz")
trades = pd.read_csv(R / "trades.csv.gz")   # the bootstrap day axis spans the whole population, as the run did
print("frozen", f0["created_utc"], "| outcome", report["created_utc"])
print("family:", report["family"])
print("verdict:", verdict["verdict"])
print("labels:", json.dumps(verdict["classifications"], indent=1))

frozen 2026-09-18T07:07:47+00:00 | outcome 2026-09-18T07:11:33+00:00
family: ['chento_BTC:F1_against', 'squeeze_bull:F1_against']
verdict: NONE PROMOTED
labels: {
 "chento_BTC:F1_against": "CONTRARY",
 "squeeze_bull:F1_against": "UNDETERMINED"
}


## 1. The preconditions, and what the family was fixed on

Eleven checks, all passing. Two are worth seeing: the new premium panel reproduces the top-anatomy stage's cache
exactly on their shared minutes, and the walker reproduces stage 1's saved walks on all 585 trades.

In [2]:
print("preconditions:", {k: pre[k]["pass"] for k in pre if k.startswith("P")})
ov = pre["P7"]["panels"]["BTCUSDT_premium_1m_panel"]["premium_overlap"]
print("\npremium panel vs the anatomy cache:", ov["minutes"], "minutes,",
      "identical" if ov["identical"] else "DIFFERENT", "| differing:", ov["differing_minutes"])
print("walker vs stage 1:", pre["P4"]["trades"], "trades, differences:", pre["P4"]["differences"])
rows = []
for pop in ("chento_BTC", "squeeze_bull", "chento_ETH", "short_squeeze"):
    for kind in ("F1_against", "F2_perp_led"):
        e = pre["P11"]["per_subpop"][pop][kind]
        rows.append({"population": pop, "kind": kind, "eligible": e["eligible_trades"],
                     "event trades": e["trades_with_event"],
                     "median elapsed (h)": round(e.get("median_elapsed_hours", float("nan")), 1),
                     "rung1": e.get("included_rung1"), "rung2": e.get("included_rung2"),
                     "rung3": e.get("included_rung3"), "decision rung": e["decision_rung"]})
pd.DataFrame(rows)

preconditions: {'P1': True, 'P2': True, 'P3': True, 'P4': True, 'P5': True, 'P6': True, 'P7': True, 'P8': True, 'P9': True, 'P10': True, 'P11': True}

premium panel vs the anatomy cache: 2472480 minutes, identical | differing: 0
walker vs stage 1: 585 trades, differences: {'i0': 0, 'x': 0, 'kind': 0, 'x_notime': 0, 'kind_notime': 0, 'level_valid': 0, 'exit_price': 0, 'exit_price_notime': 0}


,population,kind,eligible,event trades,median elapsed (h),rung1,rung2,rung3,decision rung
0,chento_BTC,F1_against,208,78,2.7,59,61,66,rung1
1,chento_BTC,F2_perp_led,208,13,6.6,5,5,10,NaN
2,squeeze_bull,F1_against,122,50,3.0,42,43,49,rung1
3,squeeze_bull,F2_perp_led,122,5,15.2,4,4,4,NaN
4,chento_ETH,F1_against,184,59,4.0,42,46,52,rung1
5,chento_ETH,F2_perp_led,184,21,21.5,8,9,12,NaN
6,short_squeeze,F1_against,71,5,1.7,0,2,1,NaN
7,short_squeeze,F2_perp_led,71,0,NaN,0,0,0,NaN


## 2. The decision lines, recomputed from the saved events

`Δ` is the continuation value at the first qualifying minute minus the matched placebo, in each strategy's own R. The
assertion below is the point of this notebook: the numbers in `report.json` are reproduced from the saved per-event
rows.

In [3]:
def recompute(subpop, kind):
    sub = events[(events["subpop"] == subpop) & (events["kind"] == kind)]
    sub = sub[np.isfinite(sub["placebo"])].sort_values("entry_ts", kind="mergesort")
    axis = M.day_axis(trades[trades["subpop"] == subpop]["entry_day"])   # same axis the outcome run used
    idx = M.block_indices(len(axis))
    return M.summarize(sub, "delta", axis, idx)

checked = []
for key in ["chento_BTC:F1_against", "squeeze_bull:F1_against", "chento_ETH:F1_against"]:
    pop, kind = key.split(":")
    t = report["tests"][key]; saved = t["lines"][t["decision_rung"]]
    mine = recompute(pop, kind)
    assert mine["n"] == saved["n"], (key, mine["n"], saved["n"])
    assert abs(mine["mean"] - saved["mean"]) < 1e-9, key
    assert max(abs(a - b) for a, b in zip(mine["ci95"], saved["ci95"])) < 1e-9, key
    checked.append({"test": key, "n": saved["n"], "Δ": round(saved["mean"], 3),
                    "95% low": round(saved["ci95"][0], 3), "95% high": round(saved["ci95"][1], 3),
                    "first half": round(saved["first_half"], 2), "second half": round(saved["second_half"], 2),
                    "holding after": round(saved["mean_cv_at_event"], 3),
                    "matched highs": round(saved["mean_placebo"], 3),
                    "Holm p": None if t.get("holm_p") is None else round(t["holm_p"], 3),
                    "label": t["classification"]})
print("recomputed from events.csv.gz and matched report.json exactly")
pd.DataFrame(checked)

recomputed from events.csv.gz and matched report.json exactly


,test,n,Δ,95% low,95% high,first half,second half,holding after,matched highs,Holm p,label
0,chento_BTC:F1_against,59,1.068,0.100,2.034,1.27,0.86,1.307,0.239,0.986,CONTRARY
1,squeeze_bull:F1_against,42,-0.036,-0.261,0.206,0.03,-0.10,0.338,0.374,0.740,UNDETERMINED
2,chento_ETH:F1_against,42,0.617,-0.357,1.532,0.06,1.17,1.201,0.584,NaN,DESCRIPTIVE


The Holm p values test whether `Δ` is **negative**, which is what an exit needs. Neither family test is close. On
chento BTC the interval lies above zero instead: holding after a perpetual-led high beat holding at matched highs by
about 1.07 R.

## 3. Every placebo, and the leg that separates

If the result were an artefact of one matching rule it would move when the rule changes. It does not.

In [4]:
def lines_of(key):
    t = report["tests"][key]; rung = t["decision_rung"]
    order = ["rung1", "rung2", "rung3", "S1", "S2", "S3", "S4", "S5", f"rvol_{rung}", f"rsession_{rung}",
             f"shared_{rung}"]
    return {n.replace(f"_{rung}", ""): (round(t["lines"][n]["mean"], 3) if t["lines"].get(n, {}).get("n") else None)
            for n in order if n in t["lines"]}
pd.DataFrame({k: lines_of(k) for k in ["chento_BTC:F1_against", "squeeze_bull:F1_against", "chento_ETH:F1_against"]})

,chento_BTC:F1_against,squeeze_bull:F1_against,chento_ETH:F1_against
rung1,1.068,-0.036,0.617
rung2,0.916,-0.036,0.661
rung3,0.653,0.044,0.696
S1,0.811,-0.053,0.590
S2,0.655,-0.008,0.636
S3,1.148,0.113,0.856
S4,1.340,-0.216,0.584
S5,1.044,-0.024,0.621
rvol,0.987,-0.122,0.903
rsession,0.986,-0.129,-0.570


In [5]:
dec = []
for pop in ("chento_BTC", "squeeze_bull"):
    for kind in ("F1_against", "F1_spot_confirmed", "F1_flow_only", "F1_flow_only_ctrl", "F1_prem_only",
                 "F1_prem_only_ctrl", "F1_mirror", "F1_oi_up", "F1_oi_down", "F1_against_usdt_fdusd",
                 "F2_perp_extreme", "F3"):
        t = report["tests"].get(f"{pop}:{kind}")
        line = t["lines"].get(t["decision_rung"] or "rung1", {}) if t else {}
        if line.get("n"):
            dec.append({"population": pop, "kind": kind, "n": line["n"], "Δ": round(line["mean"], 3),
                        "95% low": round(line["ci95"][0], 3), "95% high": round(line["ci95"][1], 3)})
pd.DataFrame(dec)

,population,kind,n,Δ,95% low,95% high
0,chento_BTC,F1_against,59,1.068,0.100,2.034
1,chento_BTC,F1_spot_confirmed,104,-0.382,-1.003,0.253
2,chento_BTC,F1_flow_only,70,1.046,0.220,1.888
3,chento_BTC,F1_flow_only_ctrl,99,-1.064,-1.761,-0.347
4,chento_BTC,F1_prem_only,85,-0.070,-0.771,0.653
5,chento_BTC,F1_prem_only_ctrl,120,0.124,-0.454,0.689
6,chento_BTC,F1_mirror,52,0.135,-0.370,0.736
7,chento_BTC,F1_oi_up,38,1.259,-0.041,2.479
8,chento_BTC,F1_oi_down,23,0.728,-0.568,1.987
9,chento_BTC,F1_against_usdt_fdusd,32,1.642,0.574,2.679


The venue leg is the one that separates, and it separates the wrong way for an exit: on chento BTC, highs where spot
lagged ran on (`F1_flow_only` +1.05) while highs where spot confirmed did not (`F1_flow_only_ctrl` −1.06). The premium
leg alone carries nothing. The site-derived parameterisation (`F3`) and the USDT-plus-FDUSD composition line point the
same way, and squeeze_bull shows none of it.

## 4. The two tables computed after the verdict

The target-minute channel was the main structural worry: the event sits at a new running extreme, which is the kind of
minute that fills a target, so a trade whose first such minute is its exit minute would be dropped into the control
pool. It does not happen.

In [6]:
ex = json.loads((R / "exploratory_reported_tables.json").read_text())
ch = {p: {"pattern at the exit minute": v["pattern_holds_at_exit_minute"]["F1_against"],
          "and it was the trade's first": v["first_pattern_minute_is_exit"]["F1_against"]}
      for p, v in ex["target_minute_channel"].items()}
display(pd.DataFrame(ch).T)
bal = {k: {"volume ratio (event/control)": f'{v["median_vr"]["event"]:.2f} / {v["median_vr"]["control"]:.2f}',
           "ordinal (event/control)": f'{v["ordinal_of_extreme"]["event"][1]:.0f} / {v["ordinal_of_extreme"]["control"][1]:.0f}',
           "weekend (event/control)": f'{v["weekend_share"]["event"]:.2f} / {v["weekend_share"]["control"]:.2f}'}
       for k, v in ex["covariate_balance"].items()}
pd.DataFrame(bal).T

,pattern at the exit minute,and it was the trade's first
chento_BTC,0,"{'target': 0, 'stop': 0, 'time': 0, 'other': 0}"
chento_ETH,0,"{'target': 0, 'stop': 0, 'time': 0, 'other': 0}"
squeeze_bull,1,"{'target': 0, 'stop': 0, 'time': 0, 'other': 0}"
short_squeeze,1,"{'target': 1, 'stop': 0, 'time': 0, 'other': 0}"


,volume ratio (event/control),ordinal (event/control),weekend (event/control)
chento_BTC:F1_against,0.73 / 0.72,10 / 10,0.42 / 0.34
chento_BTC:F1_spot_confirmed,0.66 / 0.65,8 / 8,0.40 / 0.42
chento_ETH:F1_against,0.74 / 0.73,10 / 9,0.24 / 0.36
chento_ETH:F1_spot_confirmed,0.77 / 0.73,6 / 6,0.30 / 0.25
squeeze_bull:F1_against,0.55 / 0.73,11 / 12,0.31 / 0.27
squeeze_bull:F1_spot_confirmed,0.75 / 0.58,10 / 8,0.26 / 0.36


## 5. Amendment A26: the alignment check, corrected after it failed

The precondition originally correlated the premium **level** against the perpetual-over-spot **level**. Two
autocorrelated level series correlate about 0.92 at every lag from −3 to +3, so the winning lag is noise. The cell
below shows that directly, beside the returns half of the same check, which is decisive.

In [7]:
lags = pre["P5"]["lags"]
show = {}
for key in ["BTC:2020:returns", "BTC:2020:premium_levels_reported", "BTC:2020:premium", "BTC:2024:premium"]:
    if key in lags:
        show[key] = {f"lag {k}": (None if v is None else round(v, 3)) for k, v in lags[key]["table"].items()}
        show[key]["peak"] = lags[key]["peak_lag"]; show[key]["decided"] = lags[key]["decided"]
display(pd.DataFrame(show).T)
off = {k: v["peak_lag"] for k, v in lags.items() if v["decided"] and v["peak_lag"] != 0}
print("decided lines peaking off lag 0:", off or "none")
print("2020 is reported, never decided: the earliest minute any population reads is 2021-04-16 23:15 UTC")

,lag -3,lag -2,lag -1,lag 0,lag 1,lag 2,lag 3,peak,decided
BTC:2020:returns,0.004,-0.02,0.006,0.974,-0.02,-0.016,0.008,0,False
BTC:2020:premium_levels_reported,0.916,0.927,0.956,0.929,0.918,0.91,0.907,-1,False
BTC:2020:premium,-0.035,-0.159,0.484,-0.13,-0.038,-0.04,0.039,-1,False
BTC:2024:premium,0.003,-0.036,-0.098,0.325,-0.138,-0.014,0.0,0,True


decided lines peaking off lag 0: none
2020 is reported, never decided: the earliest minute any population reads is 2021-04-16 23:15 UTC


## 6. Verdict

**NONE PROMOTED.** Neither family test is INFORMATIVE, so no stage 2 exit arm may be written on these events, and
nothing in production changes. Item F is closed as an exit signal.

What survives is a hypothesis pointing the other way and only on chento: highs where Binance spot takers lag ran on,
while highs where spot confirmed did not. That is an entry-side question, it is not protected by the family
correction, it is absent on squeeze_bull, and it would need its own pre-registration.

In [8]:
print(json.dumps(verdict, indent=1))

{
 "created_utc": "2026-09-18T07:11:32+00:00",
 "promoted": [],
 "per_kind": {
  "F1_against": {
   "informative_family_tests": [],
   "replication": "not evaluated"
  },
  "F2_perp_led": {
   "informative_family_tests": [],
   "replication": "not evaluated"
  }
 },
 "classifications": {
  "chento_BTC:F1_against": "CONTRARY",
  "squeeze_bull:F1_against": "UNDETERMINED"
 },
 "verdict": "NONE PROMOTED",
 "permits": "nothing in production; a promotion permits only a separate stage 2 pre-registration"
}
